Para entender a fundo os tipos de compressão no Apache Spark e no ecossistema de Big Data, precisamos olhar para o equilíbrio entre duas forças: **Poder de Compressão** (o quanto reduz o arquivo) vs. **Velocidade** (o quão rápido o Spark consegue ler/escrever) e a capacidade de divisão (**Splittability**).

Aqui está o raio-x dos principais algoritmos de compressão utilizados:

---

## Os Principais Algoritmos de Compressão

### 1. Snappy

Desenvolvido pelo Google, o Snappy não foca em conseguir o menor arquivo possível, mas sim em ser **extremamente rápido**.

* **Foco:** Velocidade de CPU.
* **Splittable (Divisível)?** Não por si mesmo, mas quando usado dentro de formatos de arquivo como **Parquet** ou **ORC**, o Spark consegue dividir o processamento perfeitamente.
* **Quando usar:** É o padrão da indústria para o dia a dia no Spark. Excelente para tabelas analíticas que são consultadas a todo momento.

### 2. Zstandard (zstd)

Criado pelo Facebook, o Zstandard é a tecnologia mais moderna dessa lista e oferece o melhor dos dois mundos.

* **Foco:** Flexibilidade (ótimo equilíbrio entre tamanho e velocidade).
* **Splittable (Divisível)?** Sim, nativamente em várias implementações e excelente com Parquet.
* **Quando usar:** Está se tornando o substituto do Snappy e do Gzip. Use quando quiser arquivos menores que o Snappy, mas sem o impacto de lentidão que o Gzip traria.

### 3. Gzip

O algoritmo mais popular da internet. Ele aperta muito os dados, reduzindo o armazenamento ao máximo.

* **Foco:** Taxa de compressão máxima (redução de espaço).
* **Splittable (Divisível)?** **Não.** Este é o seu maior problema no Spark.
* **Quando usar:** Apenas para dados frios (arquivamento de longo prazo que raramente será lido) ou para exportar relatórios finais pequenos para outras equipes.

### 4. Bzip2

Oferece uma taxa de compressão ainda maior que o Gzip, mas exige muito uso de processador (CPU).

* **Foco:** Alta compressão mantendo a capacidade de divisão.
* **Splittable (Divisível)?** **Sim.**
* **Quando usar:** Se você for *obrigado* a processar arquivos de texto gigantes (como CSV ou JSON de muitos Gigabytes) e precisa que o Spark processe em paralelo, o Bzip2 permite isso.

---

## O Conceito Crucial: O que é um arquivo *Splittable*?

No ecossistema Spark (Big Data), o conceito de ser **divisível (*splittable*)** é mais importante do que o tamanho final do arquivo.

* **Se o arquivo É divisível:** O Spark quebra um único arquivo grande em várias partes menores e joga cada parte para um "trabalhador" (Executor) diferente processar ao mesmo tempo (Processamento Paralelo).
* **Se o arquivo NÃO É divisível (como CSV + Gzip):** Apenas um único Executor terá que ler o arquivo inteiro do início ao fim, deixando todos os outros computadores do cluster ociosos. Isso cria um gargalo gigante de performance.

---

## Resumo de Performance vs. Espaço

Podemos resumir o comportamento dos algoritmos da seguinte forma:

* **Mais Veloz (Menos compressão):** Snappy
* **Equilíbrio Perfeito:** Zstandard (zstd)
* **Mais Leve no Disco (Muito lento):** Gzip / Bzip2

In [0]:
path = "/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet"
data = spark\
        .read\
        .format("parquet")\
        .options(
            inferSchema = True,
            header = True
        )\
        .load(path)

In [0]:
data.limit(10).display()

id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals
0,2017-11-01T14:45:41.000Z,Marta,Jesus,null,9 9102-7834,Brasil,Acre,null,null,Conjunto 16
1,2017-10-16T00:50:39.000Z,Luana,Almeida,null,9 7328-8718,Brasil,Rio Grande do Sul,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
2,2018-06-16T17:51:29.000Z,Frida,Mendes,frida@meu_email.com,9 5906-7552,Brasil,São Paulo,Avenida 59 do Estado São Paulo,534.0,null
3,2018-01-17T03:02:58.000Z,Daniela,Avelino,daniela@exemplo.com,9 4642-9486,Brasil,Mato Grosso,null,null,null
4,2018-08-06T07:24:16.000Z,Romário,Teixeira,null,9 3093-6522,Brasil,Bahia,Praça 56 do Estado Bahia,191.0,Apto 12
5,2018-01-05T17:20:49.000Z,Marcelo,Barroso,null,9 2830-2088,Brasil,Rio Grande do Sul,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
6,2018-06-18T11:17:42.000Z,Cristiano,Elísio,cristiano@exemplo.com,9 3532-8404,Brasil,Goiás,Rua 78 do Estado Goiás,877.0,Apto 14
7,2018-02-08T12:36:09.000Z,Everton,Barbosa,everton@meu_email.com,9 2553-4087,Brasil,Distrito Federal,Avenida 86 do Estado Distrito Federal,864.0,Apto 14
8,2017-12-16T20:47:03.000Z,Gabriela,Alves,gabriela@exemplo.com,9 1353-8433,Brasil,Santa Catarina,null,null,null
9,2018-11-11T11:48:41.000Z,Luan,Dias,luan@exemplo.com,9 2417-3678,Brasil,Distrito Federal,Avenida 54 do Estado Distrito Federal,889.0,Conjunto 14


# Particionamento

O **Particionamento** no Apache Spark é uma das técnicas mais importantes para otimizar o processamento e o armazenamento de grandes volumes de dados.

Ele consiste em dividir um grande conjunto de dados em partes menores (partições), permitindo que o Spark processe essas partes em paralelo usando múltiplos nós (computadores) do cluster.

Existem dois contextos diferentes quando falamos de particionamento: **em memória** (durante o processamento) e **em disco** (na hora de salvar). Vamos entender ambos.

---

## 1. Particionamento em Memória (Tuning de Performance)

Quando o Spark lê ou processa dados, ele os divide em partições na memória RAM dos *Executors*. Cada partição é processada por uma única *Task* (uma linha de execução na CPU).

### O problema do desequilíbrio (Data Skew)

Idealmente, as partições devem ter tamanhos iguais (geralmente entre **100 MB e 128 MB** cada).
Se uma partição ficar com 5 GB e as outras com 50 MB, o Spark sofrerá com o efeito *Skew* (um nó trabalhando demais enquanto os outros ficam ociosos), o que pode causar erros de `OutofMemory`.

### Como controlar no PySpark: `repartition()` vs `coalesce()`

Para ajustar o número de partições em memória durante o código, usamos dois comandos essenciais:

* **`df.repartition(n)`**: Aumenta ou diminui o número de partições. Ele faz um **Shuffle** completo dos dados pela rede (embaralha tudo para redistribuir igualmente). É uma operação cara, mas garante partições de tamanhos iguais.
* **`df.coalesce(n)`**: **Apenas diminui** o número de partições. Ele não faz um Shuffle completo; apenas junta partições vizinhas no mesmo nó. É extremamente rápido e eficiente para reduzir arquivos pequenos antes de salvar.

---

## 2. Particionamento em Disco (Otimização de Armazenamento/Leitura)

Quando salvamos um DataFrame no Data Lake (seja em Parquet, CSV ou JSON), podemos criar uma estrutura de diretórios baseada nos valores de uma ou mais colunas. Isso é feito usando o `.partitionBy()`.

### Como funciona no código:

```python
# Salvando dados de vendas particionados por Ano e Mês
df.write.mode("overwrite") \
  .partitionBy("ano", "mes") \
  .parquet("/mnt/datalake/vendas")
```

Isso criará uma estrutura de pastas no seu armazenamento como esta:

```text
/mnt/datalake/vendas/
  ├── ano=2025/
  │    ├── mes=01/
  │    │    └── part-00000.snappy.parquet
  │    └── mes=02/
  │         └── part-00000.snappy.parquet
  └── ano=2026/
       └── mes=01/
            └── part-00000.snappy.parquet

```

### A Grande Vantagem: *Partition Pruning* (Poda de Partição)

Quando alguém fizer uma consulta filtrando por um ano específico, o Spark vai **direto na pasta daquele ano** e ignora completamente o resto do Data Lake.


## Boas Práticas e Cuidados com o Particionamento em Disco

> ⚠️ **Cuidado com a Alta Cardinalidade (O maior erro no Spark)**
> Nunca particione seus dados em disco por colunas com muitos valores únicos (como `id_usuario`, `CPF`, `data_hora_exata`). Isso criará milhões de pastas pequenas no seu armazenamento, gerando o problema dos **"Small Files"** (arquivos pequenos), o que destrói a performance do Spark ao ler os metadados.

* **O que escolher como chave de partição em disco?** Colunas com baixa cardinalidade, mas que dividam bem os dados (ex: `ano`, `mes`, `regiao`, `status_pedido`).
* **Regra de ouro:** Cada partição final no disco deve ter, idealmente, entre **100 MB e 1 GB**. Se as pastas estiverem ficando com arquivos de poucos KBs, você está particionando demais.

In [0]:
from pyspark.sql.functions import year

In [0]:
data = data.withColumn("ano", year(data["created_at"]))

In [0]:
data\
    .write\
    .format("parquet")\
    .options(compression="ZSTD")\
    .partitionBy("ano", "state")\
    .mode("overwrite")\
    .save("/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet.partition")

In [0]:
spark\
    .read\
    .format("parquet")\
    .options(
        inferSchema = True,
        header = True
    )\
    .load("/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet.partition/ano=2017/state=Acre/").limit(10).display()

id,created_at,first_name,last_name,email,cell_phone,country,street,number,additionals
92,2017-10-16T16:29:15.000Z,Beatriz,Bueno,beatriz@teste.com,9 4435-2960,Brasil,Rua 71 do Estado Acre,978.0,Apto 24
0,2017-11-01T14:45:41.000Z,Marta,Jesus,null,9 9102-7834,Brasil,null,null,Conjunto 16


In [0]:
spark\
    .read\
    .format("parquet")\
    .options(
        inferSchema = True,
        header = True
    )\
    .load("/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet.partition/ano=2017/").limit(10).display()

id,created_at,first_name,last_name,email,cell_phone,country,street,number,additionals,state
78,2017-11-02T22:13:22.000Z,Laura,Queiroz,laura@exemplo.com,9 6338-8617,Brasil,Praça 19 do Estado São Paulo,878.0,Conjunto 3,São Paulo
59,2017-12-03T16:24:32.000Z,Stephanie,Freire,stephanie@usuario.com,9 3256-1853,Brasil,Rua 52 do Estado São Paulo,34.0,Apto 24,São Paulo
20,2017-12-22T10:51:28.000Z,Adalberto,Silva,adalberto@usuario.com,9 1338-1447,Brasil,Rua 57 do Estado São Paulo,851.0,Conjunto 20,São Paulo
71,2017-12-12T16:16:52.000Z,Gerson,Barroso,gerson@exemplo.com,9 5719-4227,Brasil,null,null,Apto 2,Pernambuco
51,2017-12-26T15:38:15.000Z,Caio,Mendes,caio@teste.com,9 6437-5008,Brasil,Rua 34 do Estado Pernambuco,931.0,null,Pernambuco
17,2017-10-19T20:18:14.000Z,João,Magalhães,joão@exemplo.com,9 5995-1272,Brasil,Rua 20 do Estado Pernambuco,781.0,Apto 17,Pernambuco
79,2017-12-13T08:41:05.000Z,Hélen,Souza,hélen@teste.com,9 9974-1066,Brasil,Avenida 20 do Estado Paraná,661.0,Conjunto 13,Paraná
33,2017-12-28T12:51:34.000Z,João,da Silva,joão@usuario.com,9 2813-8103,Brasil,Avenida 27 do Estado Paraná,240.0,Apto 6,Paraná
39,2017-11-15T07:17:25.000Z,Caio,Batista,caio@usuario.com,9 4761-6542,Brasil,Avenida 87 do Estado Maranhão,656.0,Apto 25,Maranhão
16,2017-12-12T21:29:54.000Z,Laura,Santos,laura@exemplo.com,9 7802-6152,Brasil,Rua 57 do Estado Maranhão,213.0,null,Maranhão


In [0]:
data = spark\
    .read\
    .format("parquet")\
    .options(
        inferSchema = True,
        header = True
    )\
    .load("/Volumes/learn_databricks/schema/volume/Clientes/parquet/Clientes.parquet.partition")

In [0]:
data.select("*").show()

+---+-------------------+----------+---------+--------------------+-----------+-------+--------------------+------+-----------+----+-----------------+
| id|         created_at|first_name|last_name|               email| cell_phone|country|              street|number|additionals| ano|            state|
+---+-------------------+----------+---------+--------------------+-----------+-------+--------------------+------+-----------+----+-----------------+
| 76|2018-04-16 04:53:31|   Romário|     Rosa|romário@meu_email...|9 8710-8305| Brasil|Praça 75 do Estad...| 110.0|Conjunto 10|2018|Rio Grande do Sul|
| 72|2018-01-29 03:19:24|    Neymar|   Elísio|    neymar@teste.com|9 5373-1785| Brasil|Rua 49 do Estado ...| 929.0|       NULL|2018|Rio Grande do Sul|
| 30|2018-06-08 07:49:50|   Gabriel| Teixeira| gabriel@exemplo.com|9 7419-5325| Brasil|Avenida 7 do Esta...|  94.0|       NULL|2018|Rio Grande do Sul|
| 13|2018-08-27 21:15:17|  Carolina| Monteiro|  carolina@teste.com|9 1226-4267| Brasil|Rua 4 d